# Intro 09 — From Probability to Statistics: Sampling and Estimation

Practice notebook for the **"From Probability to Statistics: Sampling and Estimation"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Population vs. sample

Probability reasons *forward*: given a known distribution, what data do we expect?
Statistics reasons *backward*: given observed data, what can we say about the
distribution that generated it? The bridge is the act of **sampling**.

- A **population** is the entire set of objects/observations of interest, e.g.
  all daily returns of a stock over all time. Its summary numbers are called
  **parameters** and are conventionally Greek: \(\mu\), \(\sigma^2\).
- A **sample** is a finite subset we actually observe, e.g. the last 100 days.
  Its summary numbers are called **statistics** and are conventionally Roman:
  \(\bar{X}\), \(s^2\).

We treat the population as a probability distribution with fixed but unknown
parameters, and the sample as a set of i.i.d. draws
\(X_1, \dots, X_n \sim F(\mu, \sigma^2)\). Statistics uses the sample to *infer*
the parameters.

Below we build a known population, draw samples, and visualize the distinction.


In [ ]:
# A "population" = a known probability distribution.
# We pretend we know it, so we can verify our estimators against the truth.
mu_true = 2.0       # population mean
sigma_true = 1.5    # population standard deviation
pop = stats.norm(loc=mu_true, scale=sigma_true)

# The "population" is conceptually infinite; we can sample as much as we want.
# Draw a single sample of size n (what an analyst would actually see).
n = 50
rng = np.random.default_rng(42)
sample = rng.normal(loc=mu_true, scale=sigma_true, size=n)

print(f"Population parameters (unknown to the analyst): mu = {mu_true}, sigma = {sigma_true}")
print(f"Sample size n = {n}")
print(f"Sample mean  x_bar = {sample.mean():.4f}")
print(f"Sample sd    s     = {sample.std(ddof=1):.4f}")

# Visualize: the population density vs. the observed sample
xs = np.linspace(mu_true - 4*sigma_true, mu_true + 4*sigma_true, 400)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xs, pop.pdf(xs), color="crimson", lw=2, label=rf"population pdf  ($\mu={mu_true},\ \sigma={sigma_true}$)")
ax.hist(sample, bins=12, density=True, alpha=0.4, color="steelblue", edgecolor="white", label=f"sample histogram  (n={n})")
ax.axvline(sample.mean(), color="steelblue", ls="--", lw=1.5, label=rf"$\bar{{x}} = {sample.mean():.3f}$")
ax.axvline(mu_true, color="crimson", ls=":", lw=1.5, label=rf"true $\mu = {mu_true}$")
ax.set_xlabel("x"); ax.set_ylabel("density")
ax.set_title("Population vs. sample")
ax.legend()
plt.show()


---
## Part 2 — Point estimators: the sample mean is unbiased

A **point estimator** \(\hat{\theta}\) is a rule (a function of the data) used to
estimate a parameter \(\theta\). Two qualities make an estimator good:

- **Low bias**: \(E[\hat{\theta}] \approx \theta\). If \(E[\hat{\theta}] = \theta\)
  exactly, the estimator is **unbiased**.
- **Low variance**: the estimator shouldn't swing wildly from sample to sample.

For the population mean, the natural estimator is the **sample mean**

$$
\bar{X} = \frac{1}{n}\sum_{i=1}^{n} X_i.
$$

Linearity of expectation gives

$$
E[\bar{X}] = \frac{1}{n}\sum_{i=1}^{n} E[X_i] = \mu,
$$

so \(\bar{X}\) is an **unbiased** estimator of \(\mu\), and independence gives

$$
\mathrm{Var}(\bar{X}) = \frac{1}{n^2}\sum_{i=1}^{n} \mathrm{Var}(X_i) = \frac{\sigma^2}{n}.
$$

We verify both facts by Monte Carlo: draw thousands of samples, compute
\(\bar{X}\) for each, and look at the distribution of those means.


In [ ]:
# Monte Carlo: repeat the sampling many times and collect the sample means.
n = 30
n_reps = 20_000
rng = np.random.default_rng(42)

# Each row is one sample of size n; we compute the mean of each row.
samples = rng.normal(loc=mu_true, scale=sigma_true, size=(n_reps, n))
x_bars = samples.mean(axis=1)

print(f"True population mean      mu      = {mu_true}")
print(f"Monte Carlo E[x_bar]               = {x_bars.mean():.5f}   (should be ~ mu)")
print(f"Theoretical Var(x_bar) = sig^2/n   = {sigma_true**2 / n:.5f}")
print(f"Monte Carlo Var(x_bar)             = {x_bars.var(ddof=1):.5f}   (should match)")

# Plot the Monte Carlo distribution of x_bar vs. the theoretical N(mu, sig^2/n)
xs = np.linspace(mu_true - 4*sigma_true/np.sqrt(n), mu_true + 4*sigma_true/np.sqrt(n), 400)
theo = stats.norm(loc=mu_true, scale=sigma_true/np.sqrt(n))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(x_bars, bins=60, density=True, alpha=0.5, color="steelblue", edgecolor="white", label=rf"MC distribution of $\bar{{X}}$ (n={n})")
ax.plot(xs, theo.pdf(xs), color="crimson", lw=2, label=rf"theory: $\mathcal{{N}}(\mu,\ \sigma^2/n)$")
ax.axvline(mu_true, color="black", ls=":", lw=1.5, label=rf"$\mu = {mu_true}$")
ax.set_xlabel(r"$\bar{X}$"); ax.set_ylabel("density")
ax.set_title(rf"Sample mean is unbiased: $E[\bar{{X}}]=\mu$,  $\mathrm{{Var}}(\bar{{X}})=\sigma^2/n$ ({n_reps:,} reps)")
ax.legend()
plt.show()


---
## Part 3 — Sampling distribution and the CLT

The **sampling distribution** of an estimator is the distribution it would have
if we repeated the sampling process many times. For the sample mean the
**Central Limit Theorem** says: for large \(n\),

$$
\bar{X} \;\dot\sim\; \mathcal{N}\!\left(\mu,\ \frac{\sigma^2}{n}\right),
$$

*regardless* of the shape of the underlying population (provided it has finite
variance). Two consequences we'll watch:

1. The sampling distribution is **centered** at \(\mu\) (unbiased) for any \(n\).
2. Its spread shrinks like \(1/\sqrt{n}\) — averaging more data buys precision.

We use a deliberately **non-normal** population (an exponential, which is
right-skewed) so we can see the CLT kick in as \(n\) grows.


In [ ]:
# A clearly non-normal population: Exponential(rate=1) -> mean=1, var=1
lam = 1.0
mu_exp = 1.0 / lam
sigma_exp = 1.0 / lam
n_reps = 30_000
rng = np.random.default_rng(42)

# Show the population shape once
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pop_xs = np.linspace(0, 6, 400)
axes[0].plot(pop_xs, stats.expon(scale=1/lam).pdf(pop_xs), color="crimson", lw=2)
axes[0].hist(rng.exponential(1/lam, 100_000), bins=80, density=True, alpha=0.4, color="steelblue", edgecolor="white")
axes[0].set_title(r"Population: $\mathrm{Exp}(1)$ — skewed, $\mu=1$, $\sigma^2=1$")
axes[0].set_xlabel("x"); axes[0].set_ylabel("density")

# Sampling distribution of x_bar for increasing n
ns = [1, 2, 5, 30, 200]
colors = plt.cm.viridis(np.linspace(0, 0.85, len(ns)))
for n, c in zip(ns, colors):
    # draw n_reps samples each of size n
    means = rng.exponential(1/lam, size=(n_reps, n)).mean(axis=1)
    axes[1].hist(means, bins=80, density=True, alpha=0.45, color=c, label=rf"$n={n}$")

# Overlay the CLT prediction for the largest n
n_big = ns[-1]
clt_xs = np.linspace(mu_exp - 4*sigma_exp/np.sqrt(n_big), mu_exp + 4*sigma_exp/np.sqrt(n_big), 400)
axes[1].plot(clt_xs, stats.norm(loc=mu_exp, scale=sigma_exp/np.sqrt(n_big)).pdf(clt_xs),
            color="black", lw=2, ls="--", label=rf"CLT $\mathcal{{N}}(\mu,\sigma^2/n)$, $n={n_big}$")
axes[1].axvline(mu_exp, color="black", ls=":", lw=1)
axes[1].set_title(r"Sampling distribution of $\bar{X}$ concentrates around $\mu$")
axes[1].set_xlabel(r"$\bar{X}$"); axes[1].set_ylabel("density")
axes[1].legend()

plt.tight_layout()
plt.show()

print("As n grows, the sampling distribution becomes bell-shaped (CLT)")
print("and tighter (Var = sigma^2 / n). It is always centered at mu = 1.")


---
## Part 4 — Bias vs. variance: the \(n-1\) denominator for \(s^2\)

For the population variance \(\sigma^2\), two plausible estimators are

$$
\hat{\sigma}^2_{\text{MLE}} = \frac{1}{n}\sum_{i=1}^{n}(X_i-\bar{X})^2,
\qquad
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(X_i-\bar{X})^2.
$$

The MLE-style estimator uses denominator \(n\); the **sample variance** uses
\(n-1\). Which is right? Both can't be unbiased:

- Using \(\bar{X}\) (itself estimated from the same data) makes the sum of squared
  deviations *too small on average*, by a factor of \((n-1)/n\).
- Dividing by \(n-1\) corrects this, giving \(E[s^2] = \sigma^2\) — **unbiased**.
- Dividing by \(n\) gives \(E[\hat{\sigma}^2_{\text{MLE}}] = \frac{n-1}{n}\sigma^2\)
  — **biased downward**, with bias shrinking as \(n\to\infty\).

There is a **bias-variance tradeoff**: the biased estimator has smaller variance.
Monte Carlo shows both the bias (mean off by \(\sigma^2/n\)) and the smaller
spread of the \(n\)-denominator version.


In [ ]:
# Compare the two variance estimators by Monte Carlo.
n = 10
n_reps = 50_000
rng = np.random.default_rng(42)

samples = rng.normal(loc=mu_true, scale=sigma_true, size=(n_reps, n))
# Subtract the sample mean from each row, square, and sum
dev = samples - samples.mean(axis=1, keepdims=True)
sse = np.sum(dev ** 2, axis=1)

var_mle = sse / n           # denominator n   (biased)
var_unb = sse / (n - 1)     # denominator n-1 (unbiased)

true_var = sigma_true ** 2
print(f"True population variance  sigma^2 = {true_var:.4f}")
print()
print(f"E[s^2]   (denominator n-1)        = {var_unb.mean():.4f}  (unbiased)")
print(f"E[sig^2] (denominator n)          = {var_mle.mean():.4f}  (biased low)")
print(f"Theoretical bias of MLE:  -sigma^2/n = {-true_var/n:.4f}")
print(f"Observed bias of MLE:                = {var_mle.mean() - true_var:.4f}")
print()
print(f"Var(s^2)   (n-1 denom) = {var_unb.var(ddof=1):.6f}")
print(f"Var(sig^2) (n   denom) = {var_mle.var(ddof=1):.6f}  <- smaller, but biased")

# Plot both sampling distributions
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(var_mle, bins=80, density=True, alpha=0.5, color="crimson", edgecolor="white",
        label=rf"denominator $n$: biased, mean $\approx$ {var_mle.mean():.3f}")
ax.hist(var_unb, bins=80, density=True, alpha=0.5, color="steelblue", edgecolor="white",
        label=rf"denominator $n-1$: unbiased, mean $\approx$ {var_unb.mean():.3f}")
ax.axvline(true_var, color="black", ls=":", lw=2, label=rf"true $\sigma^2 = {true_var:.3f}$")
ax.set_xlabel(r"estimator value"); ax.set_ylabel("density")
ax.set_title(rf"Two variance estimators (n={n}, {n_reps:,} reps): bias vs. variance")
ax.legend()
plt.show()

# Check unbiasedness of s^2 numerically
assert abs(var_unb.mean() - true_var) < 0.02
print("s^2 with n-1 denominator is unbiased (within MC noise).")


---
## Part 5 — Standard error and the precision of \(\bar{X}\)

The standard deviation of an estimator's sampling distribution is called its
**standard error**. For the sample mean,

$$
\mathrm{SE}(\bar{X}) = \sqrt{\mathrm{Var}(\bar{X})} = \frac{\sigma}{\sqrt{n}}.
$$

In practice \(\sigma\) is unknown and we plug in \(s\), giving
\(\widehat{\mathrm{SE}}(\bar{X}) = s/\sqrt{n}\). Either way, the precision of
\(\bar{X}\) improves only as \(1/\sqrt{n}\): to halve the error we need *four
times* as much data. We measure this directly: for each \(n\), repeat the
experiment and record the standard deviation of the \(\bar{X}\) values.


In [ ]:
# Empirical standard error of x_bar vs. n, compared to sigma/sqrt(n).
ns = np.array([5, 10, 20, 50, 100, 200, 500, 1000])
n_reps = 40_000
rng = np.random.default_rng(42)

se_emp = []
se_th = []
for n in ns:
    means = rng.normal(loc=mu_true, scale=sigma_true, size=(n_reps, n)).mean(axis=1)
    se_emp.append(means.std(ddof=1))
    se_th.append(sigma_true / np.sqrt(n))

print(f"{'n':>5} {'empirical SE':>14} {'sigma/sqrt(n)':>14} {'ratio':>8}")
for n, e, t in zip(ns, se_emp, se_th):
    print(f"{n:>5} {e:>14.5f} {t:>14.5f} {e/t:>8.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: SE vs n on linear axes
axes[0].plot(ns, se_emp, "o-", color="steelblue", lw=2, label="empirical SE of $\\bar{X}$")
axes[0].plot(ns, se_th, "k--", lw=1.5, label=r"theory: $\sigma/\sqrt{n}$")
axes[0].set_xlabel("n"); axes[0].set_ylabel(r"$\mathrm{SE}(\bar{X})$")
axes[0].set_title("Standard error shrinks with n")
axes[0].legend()

# Right: log-log -> slope -1/2
axes[1].loglog(ns, se_emp, "o-", color="steelblue", lw=2, label="empirical SE")
axes[1].loglog(ns, se_th, "k--", lw=1.5, label=r"$\sigma/\sqrt{n}$ (slope $-1/2$)")
axes[1].set_xlabel("n (log)"); axes[1].set_ylabel(r"$\mathrm{SE}(\bar{X})$ (log)")
axes[1].set_title("Log-log: $1/\\sqrt{n}$ scaling")
axes[1].legend()

plt.tight_layout()
plt.show()

# Quantify the cost of precision: doubling SE->SE/2 needs 4x the data
n0, n1 = 100, 400
print(f"SE(n={n0}) ~ {sigma_true/np.sqrt(n0):.4f}")
print(f"SE(n={n1}) ~ {sigma_true/np.sqrt(n1):.4f}  (4x data -> 2x precision)")


---
**Your turn:**

- In Part 2, change `n` from 30 to 5 and to 500. How does the spread of the
  \(\bar{X}\) distribution change? Does the mean stay at \(\mu\)?
- In Part 3, replace the exponential population with a *Bernoulli* (0/1)
  population (use `rng.integers(0, 2, ...)`). Does the sampling distribution
  of \(\bar{X}\) still become normal as \(n\) grows? What is \(\mu\)?
- In Part 4, increase `n` from 10 to 100. What happens to the bias of the
  \(n\)-denominator estimator? Does "unbiased" matter less at large \(n\)?
- In Part 5, suppose you need \(\mathrm{SE}(\bar{X}) \leq 0.01\) and
  \(\sigma \approx 1.5\). What is the minimum sample size required?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Define population, sample, parameter, and statistic in your own words.
   Give a finance-flavored example of each (e.g. daily returns of a stock).

2. Show that the sample mean \(\bar{X} = \frac{1}{n}\sum X_i\) is an unbiased
   estimator of \(\mu\) and that \(\mathrm{Var}(\bar{X}) = \sigma^2/n\). Verify
   both facts numerically with a Monte Carlo using `n=40` and `20_000` reps.

3. Simulate the sampling distribution of \(\bar{X}\) for an
   \(\mathrm{Exp}(1)\) population at \(n = 1, 5, 30\). For each \(n\), overlay
   the CLT prediction \(\mathcal{N}(\mu, \sigma^2/n)\). At what \(n\) does the
   normal approximation look good?

4. Compare the two variance estimators
   \(\hat{\sigma}^2_n = \frac{1}{n}\sum(X_i-\bar{X})^2\) and
   \(s^2 = \frac{1}{n-1}\sum(X_i-\bar{X})^2\) by Monte Carlo for \(n=6\).
   Report the mean (bias) and variance of each. Which is unbiased? Which has
   smaller variance?

5. You need the standard error of \(\bar{X}\) to be at most \(0.05\). The
   population standard deviation is unknown but a pilot sample of size 25
   gives \(s = 2.0\). What is the minimum total sample size required?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Population = the full set of units of interest (e.g. *all*
daily returns of AAPL over all time); parameter = a fixed summary of the
population, e.g. the true mean return \(\mu\) or variance \(\sigma^2\).
Sample = the finite subset we actually observe (e.g. the last 100 trading
days); statistic = any function of the sample, e.g. \(\bar{X}\) or \(s^2\),
used to *estimate* the parameter.

**2.** Independence gives \(E[\bar{X}] = \frac{1}{n}\sum E[X_i] = \mu\), and
\(\mathrm{Var}(\bar{X}) = \frac{1}{n^2}\sum \mathrm{Var}(X_i) = \sigma^2/n\).

```python
rng = np.random.default_rng(42)
n, reps = 40, 20_000
mu, sig = 2.0, 1.5
xb = rng.normal(mu, sig, (reps, n)).mean(axis=1)
print(xb.mean(), xb.var(ddof=1))          # ~ 2.0, ~ sig^2/n = 0.05625
print(sig**2 / n)                         # 0.05625
```

**3.** For \(\mathrm{Exp}(1)\), \(\mu=1\), \(\sigma=1\). The normal
approximation is visibly good by \(n\approx 30\) (rule of thumb \(n\geq 30\)),
though still slightly skewed at \(n=5\).

```python
rng = np.random.default_rng(42); reps = 30_000
for n in [1, 5, 30]:
    m = rng.exponential(1.0, (reps, n)).mean(axis=1)
    xs = np.linspace(m.mean()-4*m.std(), m.mean()+4*m.std(), 400)
    plt.hist(m, bins=60, density=True, alpha=0.5, label=f"n={n}")
    plt.plot(xs, stats.norm(1, 1/np.sqrt(n)).pdf(xs), lw=2)
plt.legend(); plt.show()
```

**4.** With \(n=6\), \(\sigma^2=2.25\):

```python
rng = np.random.default_rng(42); n, reps = 6, 50_000
mu, sig = 2.0, 1.5
s = rng.normal(mu, sig, (reps, n))
dev = s - s.mean(axis=1, keepdims=True); sse = (dev**2).sum(axis=1)
print((sse/n).mean(), (sse/(n-1)).mean())     # biased low, unbiased
print((sse/n).var(ddof=1), (sse/(n-1)).var(ddof=1))  # n-denom has smaller var
```
The \(n-1\) version is unbiased (\(E[s^2]=\sigma^2\)); the \(n\) version is
biased low by \(\sigma^2/n\) but has smaller variance — a classic
bias-variance tradeoff.

**5.** \(\mathrm{SE}(\bar{X}) = \sigma/\sqrt{n} \leq 0.05\). Plugging in the
pilot estimate \(s=2.0\) for \(\sigma\) gives
\(n \geq (2.0/0.05)^2 = 1600\). A minimum of **1600** observations.


</details>
